# Full manual lineage: Bank funding

Run each stage and inspect its output before continuing. The pinned 2026-09-26 publisher response and verified rebuild are supplied. Set `USE_PINNED_RAW = False` only for a deliberate fresh-source run; its values may differ. All processing and formulas are visible in this notebook. No portfolio module, launcher, or configuration file is required.


## 1 Source

Inspect the publisher URL and query below: https://banks.data.fdic.gov/api/financials. The included raw response is a dated publisher payload with a SHA-256 digest, not the unsupported older portfolio snapshot. To acquire a new response, set `USE_PINNED_RAW = False` before running.

**Check before continuing:** publisher, selection, project folder, and pinned-response location must match this project; any new response must be preserved with its retrieval metadata.


In [5]:
import hashlib
import json
from datetime import UTC, date, datetime
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile
import numpy as np
import pandas as pd
import requests
from IPython.display import display

SLUG = "03_sec_financial_health"
FOLDER = SLUG.split("_", 1)[1]
SOURCE_URL = "https://banks.data.fdic.gov/api/financials"
SNAPSHOT_DATE_OVERRIDE = None  # Leave None for the dated publisher response.
USE_PINNED_RAW = True
ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
             if (candidate / "projects" / FOLDER / "README.md").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the extracted portfolio.")
PROJECT = ROOT / "projects" / FOLDER
RAW_DIR = PROJECT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
FETCH_LOG = []
PINNED_RESPONSES = {'9320e4d9d42503de': 'e0cfc6a868a47a684df4d839c387675542f9d821c5a24e91d9eee1c933a5c9b8'}
print("Project:", PROJECT.relative_to(ROOT).as_posix())
print("Publisher:", SOURCE_URL)
print("Pinned response folder:", RAW_DIR.relative_to(ROOT).as_posix())

"""Contemporaneous peer monitoring; scores are not default probabilities."""


import numpy as np
import pandas as pd


def monitor(frame: pd.DataFrame, minimum_peers: int = 10) -> pd.DataFrame:
    """Compute calendar-aligned changes and complete-case quarterly peer scores.

    FDIC monetary inputs are thousands of dollars. Fixed asset-size cohorts
    avoid using later outcomes to define peers. Rankings retain genuine extremes;
    rank scoring limits their influence without modifying reported balances.
    """
    if minimum_peers < 2:
        raise ValueError("At least two peers are required")
    x = frame.copy(deep=True)
    x["report_date"] = pd.to_datetime(x["report_date"], errors="raise")
    if x.duplicated(["cert", "report_date"]).any():
        raise ValueError("Bank-quarter grain is not unique")
    if not x["report_date"].dt.is_quarter_end.all():
        raise ValueError("FDIC observations must be dated at quarter end")
    x["asset_peer"] = pd.cut(
        x["assets"],
        [0, 10_000_000, 100_000_000, 250_000_000, np.inf],
        labels=["Below $10bn", "$10bn–$100bn", "$100bn–$250bn", "$250bn+"],
        right=False,
    )
    for label, periods in [("qoq", 1), ("yoy", 4)]:
        previous = x[["cert", "report_date", "deposits"]].copy()
        previous["report_date"] += pd.offsets.QuarterEnd(periods)
        previous = previous.rename(columns={"deposits": f"deposits_{label}_base"})
        x = x.merge(previous, on=["cert", "report_date"], how="left", validate="one_to_one")
        denominator = x[f"deposits_{label}_base"].where(x[f"deposits_{label}_base"] > 0)
        x[f"deposit_growth_{label}_pct"] = 100 * (x["deposits"] / denominator - 1)
    ratios = {
        "uninsured_deposit_pct": ("uninsured_deposits", "deposits"),
        "cre_to_asset_pct": ("cre_loans", "assets"),
        "equity_to_asset_pct": ("equity", "assets"),
        "noncurrent_loan_pct": ("noncurrent_loans", "loans"),
        "loan_to_deposit_pct": ("loans", "deposits"),
    }
    for name, (numerator, denominator) in ratios.items():
        x[name] = 100 * x[numerator] / x[denominator].where(x[denominator] > 0)
    x["cash_to_asset_pct"] = (
        100
        * pd.to_numeric(x.get("cash_balances", pd.Series(np.nan, index=x.index)), errors="coerce")
        / x["assets"].where(x["assets"] > 0)
    )
    peers = ["report_date", "asset_peer"]
    x["peer_count"] = x.groupby(peers, observed=True)["cert"].transform("size")
    components = {
        "deposit_pressure": -x["deposit_growth_yoy_pct"],
        "funding_structure": x["loan_to_deposit_pct"],
        "uninsured_exposure": x["uninsured_deposit_pct"],
        "cre_exposure": x["cre_to_asset_pct"],
        "credit_deterioration": x["noncurrent_loan_pct"],
        "capital_buffer": -x["equity_to_asset_pct"],
    }
    # Compare every component on the SAME complete-case peer universe.
    raw = pd.DataFrame(components)
    complete = raw.notna().all(axis=1) & np.isfinite(raw).all(axis=1)
    x["score_component_count"] = raw.notna().sum(axis=1)
    x["complete_peer_count"] = complete.groupby([x[c] for c in peers], observed=True).transform(
        "sum"
    )
    ranked = raw.where(complete).groupby([x[c] for c in peers], observed=True).rank(pct=True)
    eligible = complete & (x["complete_peer_count"] >= minimum_peers)
    equal_weights = np.repeat(1 / 6, 6)
    funding_weights = np.array([0.25, 0.20, 0.25, 0.10, 0.10, 0.10])
    x["composite_stress_score"] = (ranked @ equal_weights * 100).where(eligible)
    x["funding_weight_score"] = (ranked @ funding_weights * 100).where(eligible)
    x["weight_sensitivity_pp"] = x["funding_weight_score"] - x["composite_stress_score"]
    x["uninsured_cre_interaction"] = x["uninsured_deposit_pct"] * x["cre_to_asset_pct"] / 100
    x["watchlist_band"] = (
        pd.cut(
            x["composite_stress_score"],
            [-np.inf, 55, 75, np.inf],
            labels=["Lower relative score", "Monitor", "Review"],
        )
        .astype("string")
        .fillna("Insufficient comparable data")
    )
    x["latest_quarter_flag"] = x["report_date"].eq(x["report_date"].max()).astype(int)
    return x.sort_values(["report_date", "cert"]).reset_index(drop=True)


def threshold_sensitivity(frame: pd.DataFrame) -> pd.DataFrame:
    """Counts use the global latest quarter; stale institutions never enter it."""
    current = frame.loc[frame["latest_quarter_flag"].eq(1)]
    rows = []
    for score in ["composite_stress_score", "funding_weight_score"]:
        for threshold in [65, 70, 75, 80]:
            valid = current[score].notna()
            rows.append(
                {
                    "weighting": score,
                    "threshold": threshold,
                    "eligible_banks": int(valid.sum()),
                    "review_banks": int(current.loc[valid, score].ge(threshold).sum()),
                    "unscored_banks": int((~valid).sum()),
                }
            )
    return pd.DataFrame(rows)

def preserve_response(
    response: requests.Response,
    raw_root: Path,
    *,
    filename: str,
    publisher: str,
    **metadata: object,
) -> Path:
    response.raise_for_status()
    if Path(filename).name != filename:
        raise ValueError("Snapshot filename must be a basename")
    payload = response.content
    if not payload:
        raise ValueError("Cannot preserve an empty source snapshot")
    digest = hashlib.sha256(payload).hexdigest()
    directory = raw_root / digest
    directory.mkdir(parents=True, exist_ok=True)
    destination = directory / filename
    if destination.exists() and hashlib.sha256(destination.read_bytes()).hexdigest() != digest:
        raise ValueError("Content-addressed snapshot collision")
    if not destination.exists():
        destination.write_bytes(payload)
    record = {
        **metadata,
        "publisher": publisher,
        "direct_url": response.url,
        "sha256": digest,
        "bytes": len(payload),
        "accessed_at": datetime.now(UTC).isoformat(),
    }
    # Response headers are deliberately excluded from public provenance.
    metadata_path = directory / "metadata.json"
    if metadata_path.exists():
        saved = json.loads(metadata_path.read_text(encoding="utf-8"))
        if saved.get("sha256") != digest or saved.get("bytes") != len(payload):
            raise ValueError("Content-addressed snapshot metadata mismatch")
    else:
        metadata_path.write_text(json.dumps(record, indent=2), encoding="utf-8")
    return destination

def project_dir(slug):
    return ROOT / "projects" / slug.split("_", 1)[1]

def _get(url, **kwargs):
    prepared = requests.Request("GET", url, params=kwargs.get("params")).prepare()
    request_key = hashlib.sha256(prepared.url.encode("utf-8")).hexdigest()[:16]
    folder = RAW_DIR / "publisher_responses" / request_key
    if USE_PINNED_RAW:
        digest = PINNED_RESPONSES.get(request_key)
        if digest is None:
            raise FileNotFoundError("This publisher request has no pinned response")
        if len(digest) != 64 or any(ch not in "0123456789abcdef" for ch in digest):
            raise ValueError("Invalid pinned source checksum")
        body = (folder / f"{digest}.body").read_bytes()
        if hashlib.sha256(body).hexdigest() != digest:
            raise ValueError("Pinned publisher response checksum mismatch")
        metadata = json.loads((folder / f"{digest}.json").read_text(encoding="utf-8"))
        response = requests.Response()
        response.status_code = 200
        response.url = prepared.url
        response._content = body
    else:
        response = requests.get(url, timeout=90, **kwargs)
        response.raise_for_status()
        body = response.content
        if not body:
            raise ValueError("Publisher returned an empty response")
        digest = hashlib.sha256(body).hexdigest()
        folder.mkdir(parents=True, exist_ok=True)
        destination = folder / f"{digest}.body"
        if not destination.exists():
            destination.write_bytes(body)
        metadata = {"url": response.url, "sha256": digest, "bytes": len(body),
                    "accessed_at": datetime.now(UTC).isoformat()}
        (folder / f"{digest}.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    FETCH_LOG.append(metadata)
    return response

def parse_fdic(payload: dict) -> pd.DataFrame:
    records = [item.get("data", item) for item in payload.get("data", [])]
    raw = pd.DataFrame(records)
    if raw.empty:
        raise ValueError("FDIC returned no quarterly records")
    total = payload.get("meta", {}).get("total")
    if total is None or len(raw) != int(total):
        raise ValueError("FDIC response is truncated or lacks total-count reconciliation")
    # LNRENROT is a subset of LNRENRES, not an additional balance.
    # Investor CRE proxy excludes owner-occupied nonfarm/nonresidential loans.
    components = raw[["LNRECONS", "LNREMULT", "LNRENROT"]].apply(pd.to_numeric, errors="coerce")
    raw["cre_loans"] = components.sum(axis=1, min_count=3)
    return raw.rename(
        columns={
            "CERT": "cert",
            "REPDTE": "report_date",
            "NAME": "bank_name",
            "STALP": "state",
            "ASSET": "assets",
            "DEP": "deposits",
            "LNLSGR": "loans",
            "NETINC": "net_income",
            "ROA": "reported_roa_pct",
            "EQTOT": "equity",
            "NCLNLS": "noncurrent_loans",
            "DEPUNINS": "uninsured_deposits",
            "CHBAL": "cash_balances",
        }
    )

def _fdic() -> pd.DataFrame:

    fields = (
        "CERT,REPDTE,NAME,STALP,ASSET,DEP,LNLSGR,NETINC,ROA,EQTOT,NCLNLS,"
        "DEPUNINS,LNRECONS,LNRENRES,LNREMULT,LNRENROT,LNRENROW,CHBAL"
    )
    response = _get(
        "https://banks.data.fdic.gov/api/financials",
        params={
            "filters": "REPDTE:[20230331 TO 20251231] AND ASSET:[10000000 TO *]",
            "fields": fields,
            "limit": 10000,
            "format": "json",
        },
    )
    raw = parse_fdic(response.json())
    preserve_response(
        response,
        project_dir("03_sec_financial_health") / "data/raw",
        filename="bank_quarters.json",
        publisher="Federal Deposit Insurance Corporation",
        source_url="https://banks.data.fdic.gov/docs/",
        coverage=f"{raw.report_date.min()} through {raw.report_date.max()}",
        grain="FDIC certificate and report quarter; assets at least $10bn in that quarter",
        refresh_frequency="Quarterly, subject to revisions",
        reuse_basis="Public US government financial statistics",
        attribution="FDIC BankFind Suite",
        limitations="Amended reports and mergers affect comparisons. Cash is not a regulatory liquidity ratio. Investor CRE uses book balances, not regulatory capital definitions.",
    )
    return raw


Project: projects/sec_financial_health
Publisher: https://banks.data.fdic.gov/api/financials
Pinned response folder: projects/sec_financial_health/data/raw


## 2 Import

Run the import cell. It verifies the pinned response hash (or retrieves a new publisher response), parses it, and displays row count, columns, and a preview.

**Check before continuing:** no parse or source-integrity error; the imported table is nonempty, has the expected fields and grain, and the displayed source date matches the saved response metadata.


In [6]:
raw = _fdic()
if raw.empty:
    raise ValueError("Publisher returned no source records")
SNAPSHOT_DATE = SNAPSHOT_DATE_OVERRIDE or (
    str(FETCH_LOG[0].get("accessed_at") or "")[:10] if FETCH_LOG else date.today().isoformat()
)
raw.to_csv(RAW_DIR / "source_snapshot.csv", index=False)
print("Snapshot date:", SNAPSHOT_DATE)
print("Publisher responses:", len(FETCH_LOG))
print("Imported rows:", len(raw), "columns:", len(raw.columns))
display(raw.head(3))


Snapshot date: 2026-09-26
Publisher responses: 1
Imported rows: 1935 columns: 20


,LNRENROT,reported_roa_pct,report_date,assets,loans,net_income,deposits,bank_name,noncurrent_loans,uninsured_deposits,LNREMULT,cash_balances,LNRENRES,cert,state,LNRENROW,LNRECONS,equity,ID,cre_loans
0,4203586.0,0.782040,20230331,31565171,18326533,62350.0,24253099,FIRST INTERSTATE BANK,85288,7976739,1207671,1080546,7075405,1105,MT,2871819.0,1902538,3232637.0,1105_20230331,7313795.0
1,4187971.0,0.852003,20230630,30920570,18339896,134478.0,23750011,FIRST INTERSTATE BANK,92752,7760710,1349200,680451,7071177,1105,MT,2883206.0,1842973,3184837.0,1105_20230630,7380144.0
2,4153174.0,0.908982,20230930,30468800,18272400,213340.0,23787754,FIRST INTERSTATE BANK,133380,7866815,1354317,591043,7020145,1105,MT,2866971.0,1936660,3141063.0,1105_20230930,7444151.0


## 3 Transform and clean

Run the original Bank funding transformation, embedded below. Inspect the fields, grain, and missing values before continuing.

**Check before continuing: inspect the cleaned preview, required fields, record grain, types, and missing values. Stop if a key or type rule fails.**


In [7]:
from dataclasses import asdict, dataclass

import pandas as pd


@dataclass(frozen=True)
class QualityReport:
    rows: int
    columns: int
    duplicate_rows: int
    missing_cells: int
    missing_rate: float
    passed: bool
    notes: list[str]


class DataQualityError(ValueError):
    """Raised when a dataset fails the required release checks."""


def profile(df: pd.DataFrame, key: str | None = None) -> dict:
    notes: list[str] = []
    duplicate_rows = int(df.duplicated().sum())
    missing_cells = int(df.isna().sum().sum())
    total_cells = max(df.shape[0] * df.shape[1], 1)
    if df.empty:
        notes.append("Dataset is empty")
    if key:
        if key not in df:
            notes.append(f"Required key '{key}' is missing")
        else:
            if df[key].isna().any():
                notes.append(f"Key '{key}' contains missing values")
            if df[key].duplicated().any():
                notes.append(f"Key '{key}' is not unique")
    if duplicate_rows:
        notes.append(f"Found {duplicate_rows} duplicate rows")
    report = QualityReport(
        rows=len(df),
        columns=len(df.columns),
        duplicate_rows=duplicate_rows,
        missing_cells=missing_cells,
        missing_rate=round(missing_cells / total_cells, 6),
        passed=not notes,
        notes=notes,
    )
    return asdict(report)


def require_passed(report: dict, project: str) -> None:
    if not report["passed"]:
        details = "; ".join(report["notes"]) or "unspecified quality failure"
        raise DataQualityError(f"Quality checks failed for {project}: {details}")


def require_columns(df: pd.DataFrame, columns: list[str]) -> None:
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise ValueError(f"Required columns missing: {', '.join(missing)}")

def _snake(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = (
        out.columns.str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return out

def bank_financials(df: pd.DataFrame) -> pd.DataFrame:
    x = _snake(df)
    require_columns(
        x,
        [
            "cert",
            "bank_name",
            "state",
            "report_date",
            "assets",
            "deposits",
            "loans",
            "equity",
            "noncurrent_loans",
        ],
    )
    numeric = [
        "assets",
        "deposits",
        "loans",
        "equity",
        "net_income",
        "noncurrent_loans",
        "reported_roa_pct",
        "uninsured_deposits",
        "cre_loans",
        "cash_balances",
    ]
    for col in numeric:
        x[col] = pd.to_numeric(x.get(col), errors="coerce")
    x["report_date"] = pd.to_datetime(x["report_date"], errors="coerce")
    x["deposit_to_asset_pct"] = (100 * x["deposits"] / x["assets"]).round(2)
    x["loan_to_deposit_pct"] = (100 * x["loans"] / x["deposits"]).round(2)
    x["equity_to_asset_pct"] = (100 * x["equity"] / x["assets"]).round(2)
    x["noncurrent_loan_pct"] = (100 * x["noncurrent_loans"] / x["loans"]).round(2)
    x["uninsured_deposit_pct"] = (100 * x["uninsured_deposits"] / x["deposits"]).round(2)
    x["cre_to_asset_pct"] = (100 * x["cre_loans"] / x["assets"]).round(2)
    # The monitor rejects duplicate grain instead of silently choosing a record.
    # Complete quarterly cohorts prevent future observations leaking into ranks.
    x = monitor(x)
    return x[
        [
            "cert",
            "bank_name",
            "state",
            "report_date",
            *numeric,
            "deposit_to_asset_pct",
            "loan_to_deposit_pct",
            "equity_to_asset_pct",
            "noncurrent_loan_pct",
            "deposit_growth_yoy_pct",
            "uninsured_deposit_pct",
            "cre_to_asset_pct",
            "composite_stress_score",
            "watchlist_band",
            "deposit_growth_qoq_pct",
            "cash_to_asset_pct",
            "asset_peer",
            "complete_peer_count",
            "funding_weight_score",
            "uninsured_cre_interaction",
            "latest_quarter_flag",
        ]
    ].drop_duplicates(["cert", "report_date"])

clean = bank_financials(raw)
print("Clean rows:", len(clean), "columns:", len(clean.columns))
display(clean.head(3))


Clean rows: 1935 columns: 30


,cert,bank_name,state,report_date,assets,deposits,loans,equity,net_income,noncurrent_loans,...,cre_to_asset_pct,composite_stress_score,watchlist_band,deposit_growth_qoq_pct,cash_to_asset_pct,asset_peer,complete_peer_count,funding_weight_score,uninsured_cre_interaction,latest_quarter_flag
0,14,STATE STREET BANK&TRUST CO,MA,2023-03-31,287069000,230488000,34139000,26373000.0,616000.0,45000,...,1.010210,NaN,Insufficient comparable data,NaN,31.686459,$250bn+,0,NaN,0.711510,0
1,110,BANK OZK,AR,2023-03-31,28971170,22282983,22062006,4763298.0,169899.0,40120,...,51.091092,NaN,Insufficient comparable data,NaN,3.587705,$10bn–$100bn,0,NaN,19.858159,0
2,131,SEACOAST NATIONAL BANK,FL,2023-03-31,15247533,12316316,10137233,2017877.0,12671.0,51668,...,25.839957,NaN,Insufficient comparable data,NaN,5.155752,$10bn–$100bn,0,NaN,8.114155,0


## 4 Calculate

Run the original analytical formulas, embedded below. Inspect the threshold_sensitivity output before continuing.

**Check before continuing: confirm the named result table is nonempty and its labels and units match the project description. Do not treat a fresh snapshot as the historical baseline.**


In [8]:
"""Decision-specific analytical routines used by notebooks and release checks."""


import re
from dataclasses import dataclass

import numpy as np
import pandas as pd

SEED = 20260904


@dataclass(frozen=True)
class AnalysisSpec:
    group: str
    metric: str
    score: str
    entity: str
    date: str | None
    higher_is_risk: bool = True


SPECS = {
    "01_consumer_complaints": AnalysisSpec(
        "product", "timely_flag", "response_risk_score", "complaint_id", "date_received", False
    ),
    "03_sec_financial_health": AnalysisSpec(
        "bank_name", "noncurrent_loan_pct", "composite_stress_score", "cert", "report_date"
    ),
    "04_hospital_readmissions": AnalysisSpec(
        "measure_name", "shrunk_readmission_ratio", "priority_score", "facility_id", None
    ),
    "05_drug_safety": AnalysisSpec(
        "therapeutic_category",
        "shortage_age_days",
        "escalation_score",
        "package_ndc",
        "update_date",
    ),
    "07_electricity_affordability": AnalysisSpec(
        "state", "sales_yoy_pct", "load_pressure_index", "state", "period"
    ),
    "08_power_plant_emissions": AnalysisSpec(
        "fuel", "co2_lb_mwh", "transition_priority_score", "plant_id", None
    ),
    "10_semiconductor_supply_chain": AnalysisSpec(
        "metric", "yoy_pct", "pressure_zscore", "series_id", "period"
    ),
}

def data_audit(frame: pd.DataFrame, entity: str) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "check": [
                "rows",
                "columns",
                "duplicate_entity_rows",
                "missing_cells",
                "numeric_columns",
            ],
            "value": [
                len(frame),
                frame.shape[1],
                int(frame.duplicated(entity).sum()) if entity in frame else 0,
                int(frame.isna().sum().sum()),
                len(frame.select_dtypes(include="number").columns),
            ],
        }
    )

def segment_scorecard(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    grouped = (
        frame.groupby(spec.group, dropna=False)
        .agg(
            records=(spec.entity, "size"),
            metric_mean=(spec.metric, "mean"),
            metric_median=(spec.metric, "median"),
            risk_score=(spec.score, "mean"),
        )
        .reset_index()
    )
    grouped["metric_gap_vs_total"] = grouped["metric_mean"] - frame[spec.metric].mean()
    grouped["risk_rank"] = (
        grouped["risk_score"].rank(method="dense", ascending=False).astype("Int64")
    )
    return grouped.sort_values(["risk_rank", "records"], ascending=[True, False])

def bootstrap_top_gap(frame: pd.DataFrame, spec: AnalysisSpec, draws: int = 1500) -> pd.DataFrame:
    rng = np.random.default_rng(SEED)
    scorecard = segment_scorecard(frame, spec)
    focus = scorecard.iloc[0][spec.group]
    a = frame.loc[frame[spec.group] == focus, spec.metric].dropna().to_numpy(float)
    b = frame.loc[frame[spec.group] != focus, spec.metric].dropna().to_numpy(float)
    if min(len(a), len(b)) < 2:
        return pd.DataFrame(
            {"focus_group": [focus], "gap": [np.nan], "ci_low": [np.nan], "ci_high": [np.nan]}
        )
    gaps = np.empty(draws)
    for i in range(draws):
        gaps[i] = (
            rng.choice(a, len(a), replace=True).mean() - rng.choice(b, len(b), replace=True).mean()
        )
    return pd.DataFrame(
        {
            "focus_group": [focus],
            "gap": [a.mean() - b.mean()],
            "ci_low": [np.quantile(gaps, 0.025)],
            "ci_high": [np.quantile(gaps, 0.975)],
            "bootstrap_draws": [draws],
        }
    )

def feature_diagnostics(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    numeric = frame.select_dtypes(include="number").drop(columns=[spec.score], errors="ignore")
    numeric = numeric.loc[:, numeric.nunique(dropna=True) > 1]
    if spec.score not in frame or numeric.empty:
        return pd.DataFrame(columns=["feature", "spearman_with_risk", "absolute_association"])
    corr = numeric.corrwith(frame[spec.score], method="spearman").dropna()
    return (
        corr.rename("spearman_with_risk")
        .rename_axis("feature")
        .reset_index()
        .assign(absolute_association=lambda d: d["spearman_with_risk"].abs())
        .sort_values("absolute_association", ascending=False)
        .head(12)
    )

def temporal_diagnostics(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    if not spec.date or spec.date not in frame:
        return pd.DataFrame(
            columns=["period", "records", "metric_mean", "risk_score", "rolling_risk"]
        )
    x = frame.dropna(subset=[spec.date]).copy()
    x["period"] = pd.to_datetime(x[spec.date]).dt.to_period("M").dt.to_timestamp()
    out = (
        x.groupby("period")
        .agg(
            records=(spec.entity, "size"),
            metric_mean=(spec.metric, "mean"),
            risk_score=(spec.score, "mean"),
        )
        .reset_index()
        .sort_values("period")
    )
    out["rolling_risk"] = out["risk_score"].rolling(3, min_periods=1).mean()
    out["risk_change"] = out["rolling_risk"].diff()
    return out

def sensitivity(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    rows = []
    for quantile in [0.70, 0.75, 0.80, 0.85, 0.90]:
        threshold = frame[spec.score].quantile(quantile)
        flagged = frame[frame[spec.score] >= threshold]
        rows.append(
            {
                "risk_quantile": quantile,
                "threshold": threshold,
                "flagged_records": len(flagged),
                "flagged_share": len(flagged) / len(frame),
                "flagged_metric_mean": flagged[spec.metric].mean(),
            }
        )
    return pd.DataFrame(rows)

def decision_queue(frame: pd.DataFrame, spec: AnalysisSpec, limit: int = 20) -> pd.DataFrame:
    columns = [spec.entity, spec.group, spec.metric, spec.score]
    if spec.date:
        columns.insert(1, spec.date)
    columns = list(dict.fromkeys(c for c in columns if c in frame))
    return frame.nlargest(limit, spec.score)[columns].reset_index(drop=True)

def bank_scenario(frame: pd.DataFrame) -> pd.DataFrame:
    latest = frame.sort_values("report_date").groupby("cert").tail(1).copy()
    for shock in [0.05, 0.10, 0.15]:
        latest[f"loan_to_deposit_after_{int(shock * 100)}pct_outflow"] = (
            100 * latest["loans"] / (latest["deposits"] * (1 - shock))
        )
    columns = [
        "cert",
        "bank_name",
        "composite_stress_score",
        *[c for c in latest if "outflow" in c],
    ]
    return latest[columns].sort_values("composite_stress_score", ascending=False)

def run_analysis(slug: str, frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    spec = SPECS[slug]
    outputs = {
        "data_audit": data_audit(frame, spec.entity),
        "segment_scorecard": segment_scorecard(frame, spec),
        "bootstrap_gap": bootstrap_top_gap(frame, spec),
        "feature_diagnostics": feature_diagnostics(frame, spec),
        "temporal_diagnostics": temporal_diagnostics(frame, spec),
        "threshold_sensitivity": sensitivity(frame, spec),
        "decision_queue": decision_queue(frame, spec),
    }
    if slug == "01_consumer_complaints":
        outputs["response_queue_intervals"] = monthly_queue(frame)
    elif slug == "03_sec_financial_health":

        outputs["deposit_outflow_scenario"] = bank_scenario(frame)
        outputs["threshold_sensitivity"] = threshold_sensitivity(frame)
    elif slug == "04_hospital_readmissions":
        outputs["observed_rate_uncertainty"] = observed_rate_review(frame)
        outputs["shrinkage_sensitivity"] = shrinkage_sensitivity(frame)
    elif slug == "05_drug_safety":
        outputs["concentration_analysis"] = shortage_concentration(frame)
        outputs["record_hhi_sensitivity"] = concentration_sensitivity(frame)
    elif slug == "07_electricity_affordability":
        outputs["growth_scenarios"] = energy_projection(frame)
        outputs["monthly_pressure_signals"] = monthly_signals(frame)
        outputs["persistence_sensitivity"] = persistence_sensitivity(frame)
    elif slug == "08_power_plant_emissions":
        outputs["abatement_pareto"] = emissions_pareto(frame)
        outputs["peer_scenarios"] = peer_scenarios(
            frame.rename(columns={"generation_mwh": "net_generation_mwh"}).drop(
                columns=["peer_count"], errors="ignore"
            )
        )
    elif slug == "10_semiconductor_supply_chain":
        outputs["divergence_analysis"] = supply_chain_divergence(frame)
        outputs.update(pulse_tables(frame))
    return outputs

analysis_outputs = run_analysis(SLUG, clean)
print("Calculated outputs:", {name: len(frame) for name, frame in analysis_outputs.items()})
display(analysis_outputs["threshold_sensitivity"].head(5))


Calculated outputs: {'data_audit': 5, 'segment_scorecard': 182, 'bootstrap_gap': 1, 'feature_diagnostics': 12, 'temporal_diagnostics': 12, 'threshold_sensitivity': 8, 'decision_queue': 20, 'deposit_outflow_scenario': 176}


,weighting,threshold,eligible_banks,review_banks,unscored_banks
0,composite_stress_score,65,148,18,11
1,composite_stress_score,70,148,5,11
2,composite_stress_score,75,148,2,11
3,composite_stress_score,80,148,0,11
4,funding_weight_score,65,148,21,11


## 5 Check

Run every assertion and inspect the quality report. These checks apply to the current response, not to unsupplied historical downloads.

**Check before continuing:** every assertion passes and `passed` is true. Investigate missing keys, duplicates, impossible values, and material row-count changes before release.


In [9]:
KEY_COLUMNS = ["cert","report_date"]
report = profile(clean)
assert report["passed"], report["notes"]
assert clean[KEY_COLUMNS].notna().all().all(), "Missing record key"
assert not clean.duplicated(KEY_COLUMNS).any(), "Duplicate record key"
assert len(raw) > 0 and len(clean) > 0
print("Current source rows:", len(raw), "clean rows:", len(clean))
print("Quality:", report)


Current source rows: 1935 clean rows: 1935
Quality: {'rows': 1935, 'columns': 30, 'duplicate_rows': 0, 'missing_cells': 2670, 'missing_rate': 0.045995, 'passed': True, 'notes': []}


## 6 Save

Run this cell only after Check passes. It saves the checked extract, quality report, analysis tables, and source manifest. Save never writes into an existing run folder. If the dated folder already exists, it creates a new timestamp-and-random-suffix run folder, even when the source fingerprint is unchanged. This keeps the original result available for comparison. Use the exact output folder printed by Save for subsequent SQL and Tableau checks; a partial run must be investigated, not merged into a later run.

**Check before continuing:** confirm the printed folder contains the current source digest, row count, and every named output. Compare with the untouched pinned result before claiming a match.

In [ ]:
# Preserve every existing saved result, even when the source bytes are identical.
from uuid import uuid4
assert report["passed"], "Run Check successfully before Save"
assert report["rows"] == len(clean), "The checked row count changed; rerun Check"
CURRENT_SOURCE_SHA = [download_report["sha256"]] if SLUG == "01_consumer_complaints" else [item["sha256"] for item in FETCH_LOG]
OUTPUT_DIR = PROJECT / "data" / "manual_rebuild" / SNAPSHOT_DATE
if OUTPUT_DIR.exists():
    run_stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
    OUTPUT_DIR = OUTPUT_DIR.with_name(f"{SNAPSHOT_DATE}-run-{run_stamp}-{uuid4().hex[:8]}")
# Exclusive creation also stops an unexpected name collision; never merge runs.
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
clean.to_csv(OUTPUT_DIR / "tableau_extract.csv", index=False, date_format="%Y-%m-%d %H:%M:%S")
report.update({"project": SLUG, "source": "FDIC BankFind financials",
               "sample_mode": False, "snapshot_date": SNAPSHOT_DATE,
               "source_sha256": [item["sha256"] for item in FETCH_LOG]})
(OUTPUT_DIR / "quality_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
analysis_dir = OUTPUT_DIR / "analysis"
analysis_dir.mkdir(exist_ok=True)
for name, frame in analysis_outputs.items():
    frame.to_csv(analysis_dir / f"{name}.csv", index=False, date_format="%Y-%m-%d")
(analysis_dir / "manifest.json").write_text(json.dumps({
    "project": SLUG, "sample_mode": False,
    "artifacts": {name: len(frame) for name, frame in analysis_outputs.items()}
}, indent=2), encoding="utf-8")
(OUTPUT_DIR / "source_manifest.json").write_text(json.dumps({
    "source_url": SOURCE_URL, "snapshot_date": SNAPSHOT_DATE,
    "responses": FETCH_LOG
}, indent=2), encoding="utf-8")
print("Saved manual rebuild to:", OUTPUT_DIR.relative_to(ROOT).as_posix())


### Manual MySQL handoff after Save

If you are demonstrating the CDC-style database step, open this project's `sql/README.md` now. In MySQL Workbench, run `create_table.sql`, stage this run's checked `tableau_extract.csv`, run a temporary path-edited copy of `load_extract.sql`, then run `check_load.sql` and `review_queries.sql`. The SQL files create and check a mirror of the saved extract; they do not replace the notebook's calculations.

**Visible output:** Workbench shows the loaded row count, distinct source keys, missing-key checks, and a small inspection query. **Check before Show:** loaded rows equal this run's `quality_report.json` rows, distinct keys equal loaded rows, missing keys equal zero, and no unexplained load warnings remain. For a fresh response, use its own dated folder, not the pinned counts. If MySQL is unavailable, continue to Show with the notebook evidence and say explicitly that the SQL mirror was not tested.


## 7 Show

Display the rebuilt analytical table below. The packaged Tableau workbook in this project's tableau folder displays the pinned 2026-09-26 extract and is an additional saved view. Compare the saved table and dashboard before citing either; a fresh publisher response requires a deliberate workbook refresh and caption review.

**Check before continuing:** the displayed rows come from the current `analysis_outputs` and saved extract, with labels and units checked against this project's README.


In [ ]:
display(analysis_outputs["threshold_sensitivity"].head(10))
print("Tableau input:", (OUTPUT_DIR / "tableau_extract.csv").relative_to(ROOT).as_posix())


,weighting,threshold,eligible_banks,review_banks,unscored_banks
0,composite_stress_score,65,148,18,11
1,composite_stress_score,70,148,5,11
2,composite_stress_score,75,148,2,11
3,composite_stress_score,80,148,0,11
4,funding_weight_score,65,148,21,11
5,funding_weight_score,70,148,9,11
6,funding_weight_score,75,148,4,11
7,funding_weight_score,80,148,1,11


## 8 Decide

The next cell records a source-dated decision, saved evidence, and an interpretation limit for the pinned response. Review and revise both statements if you acquire a fresh response; do not carry this conclusion forward automatically.

**Final check:** the current quality check passed, each evidence file exists, and the conclusion is no stronger than the analysis supports.


In [12]:
DECISION_TEXT = "For the 2026-09-26 FDIC snapshot, prioritize institution-specific review of the 2 banks flagged at the default composite threshold of 75 among 148 scored banks. The saved threshold_sensitivity.csv and quality_report.json support this screening decision, not a prediction of failure."
INTERPRETATION_LIMIT = "The score is a stress screen, not a default probability. Missing values, peer composition, and reporting revisions limit comparisons."
EVIDENCE_FILES = ["quality_report.json", "analysis/threshold_sensitivity.csv"]
assert report["passed"], "Prior quality check must pass"
assert DECISION_TEXT.strip(), "Write a decision supported by the rebuilt evidence"
assert INTERPRETATION_LIMIT.strip(), "State an interpretation limit"
missing = [name for name in EVIDENCE_FILES if not (OUTPUT_DIR / name).is_file()]
assert not missing, f"Missing evidence files: {missing}"
print("Source snapshot:", SNAPSHOT_DATE)
print("Evidence:", EVIDENCE_FILES)
print("Decision:", DECISION_TEXT)
print("Limit:", INTERPRETATION_LIMIT)


Source snapshot: 2026-09-26
Evidence: ['quality_report.json', 'analysis/threshold_sensitivity.csv']
Decision: For the 2026-09-26 FDIC snapshot, prioritize institution-specific review of the 2 banks flagged at the default composite threshold of 75 among 148 scored banks. The saved threshold_sensitivity.csv and quality_report.json support this screening decision, not a prediction of failure.
Limit: The score is a stress screen, not a default probability. Missing values, peer composition, and reporting revisions limit comparisons.
